# Idealista API processed -> gold para Streamlit, dataset completo

Replica el flujo de `idealistaAPI_processed_to_gold.ipynb`, pero conserva la totalidad de columnas de los CSV processed y no aplica los filtros finales que eliminan filas por `precio_m2`.

**Input**
- `data/processed/idealistaAPI/total_rent_cantabria_outliers.csv`
- `data/processed/idealistaAPI/total_sale_cantabria_outliers.csv`

**Output**
- `data/gold/streamlit_rent.csv`
- `data/gold/streamlit_sale.csv`

---
## 0. Imports y rutas

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 250)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "idealistaAPI").exists():
            return candidate
    raise FileNotFoundError("No se encontro la raiz del proyecto")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT))

IN_DIR = PROJECT_ROOT / "data" / "processed" / "idealistaAPI"
RENT_IN = IN_DIR / "total_rent_cantabria_outliers.csv"
SALE_IN = IN_DIR / "total_sale_cantabria_outliers.csv"

GOLD_DIR = PROJECT_ROOT / "data" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)
RENT_OUT = GOLD_DIR / "streamlit_rent.csv"
SALE_OUT = GOLD_DIR / "streamlit_sale.csv"

MIN_OBS_MUNICIPIO = 10

print(f"Project root : {PROJECT_ROOT}")
print(f"Input rent   : {RENT_IN}")
print(f"Input sale   : {SALE_IN}")
print(f"Output rent  : {RENT_OUT}")
print(f"Output sale  : {SALE_OUT}")

Project root : /Users/sitomachucas/Documents/BezanillaSL
Input rent   : /Users/sitomachucas/Documents/BezanillaSL/data/processed/idealistaAPI/total_rent_cantabria_outliers.csv
Input sale   : /Users/sitomachucas/Documents/BezanillaSL/data/processed/idealistaAPI/total_sale_cantabria_outliers.csv
Output rent  : /Users/sitomachucas/Documents/BezanillaSL/data/gold/streamlit_rent.csv
Output sale  : /Users/sitomachucas/Documents/BezanillaSL/data/gold/streamlit_sale.csv


---
## 1. Carga completa y aliases analiticos

A diferencia del notebook original, aqui no se seleccionan columnas. Se cargan todas las columnas de entrada y se crean columnas alias con nombres limpios para reutilizar el feature engineering.

In [2]:
COL_ALIAS = {
    "price": "precio",
    "size": "superficie_construida_m2",
    "rooms": "numero_dormitorios",
    "bathrooms": "numero_banos",
    "floor": "planta",
    "exterior": "es_exterior",
    "hasLift": "tiene_ascensor",
    "parkingSpace.hasParkingSpace": "tiene_garaje",
    "newDevelopment": "obra_nueva",
    "latitude": "latitud",
    "longitude": "longitud",
    "municipality": "municipio",
    "province": "provincia",
    "district": "distrito",
    "detailedType.typology": "tipologia",
    "detailedType.subTypology": "subtipologia",
    "priceByArea": "precio_m2_raw",
}


def load_full_with_aliases(path: Path, name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"{name} cargado: {df.shape}")

    for raw_col, alias_col in COL_ALIAS.items():
        if raw_col in df.columns and alias_col not in df.columns:
            df[alias_col] = df[raw_col]

    print(f"{name} con aliases: {df.shape}")
    return df


rent_df = load_full_with_aliases(RENT_IN, "RENT")
sale_df = load_full_with_aliases(SALE_IN, "SALE")

RENT cargado: (674, 49)
RENT con aliases: (674, 66)
SALE cargado: (2532, 52)
SALE con aliases: (2532, 69)


---
## 2. Enriquecimiento con distancias minimas a POIs

In [3]:
from src.geospatial_expansion import agregar_distancias_minimas_poi

TIPOS_POI = ["playa", "supermercado", "colegio"]
DISTANCE_COLS = [
    "distancia_min_playa_km",
    "distancia_min_supermercado_km",
    "distancia_min_colegio_km",
]

rent_df = agregar_distancias_minimas_poi(rent_df, TIPOS_POI)
sale_df = agregar_distancias_minimas_poi(sale_df, TIPOS_POI)

print("Distancias RENT:")
print(rent_df[DISTANCE_COLS].describe().round(3))
print("\nDistancias SALE:")
print(sale_df[DISTANCE_COLS].describe().round(3))

Distancias RENT:
       distancia_min_playa_km  distancia_min_supermercado_km  \
count                 674.000                        674.000   
mean                    3.216                          0.918   
std                     4.679                          3.280   
min                     0.056                          0.006   
25%                     1.000                          0.108   
50%                     2.104                          0.194   
75%                     2.734                          0.384   
max                    45.219                         35.918   

       distancia_min_colegio_km  
count                   674.000  
mean                      0.919  
std                       3.237  
min                       0.013  
25%                       0.136  
50%                       0.228  
75%                       0.418  
max                      35.777  

Distancias SALE:
       distancia_min_playa_km  distancia_min_supermercado_km  \
count             

---
## 3. Feature engineering sin descartar columnas ni filas

In [4]:
UNIFAMILIAR_VALUES = {
    "chalet", "countryhouse", "singlefamily", "house",
    "villa", "townhouse", "detachedhouse", "adosado",
}


def normalize_boolean(series: pd.Series) -> pd.Series:
    true_vals = {"true", "t", "1", "yes", "y", "si", "s", "sí"}
    false_vals = {"false", "f", "0", "no", "n"}

    def _conv(value):
        if pd.isna(value):
            return 0
        if isinstance(value, (bool, np.bool_)):
            return int(value)
        if isinstance(value, (int, float, np.integer, np.floating)):
            return 0 if pd.isna(value) else int(value >= 1)
        text = str(value).strip().lower()
        if text in true_vals:
            return 1
        if text in false_vals:
            return 0
        return 0

    return series.map(_conv).astype(int)


def parse_planta(value) -> int:
    if pd.isna(value):
        return 0
    text = str(value).strip().lower()
    if text in {"bj", "bajo", "entresuelo", "sotano", "sotano", "ss", "st"}:
        return 0
    match = re.search(r"-?\d+", text)
    return int(match.group()) if match else 0


def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return radius * 2 * np.arcsin(np.sqrt(a))


def add_dummies_preserving_source(df: pd.DataFrame, col: str) -> pd.DataFrame:
    values = df[col].fillna("desconocido").astype(str).str.strip().replace("", "desconocido")
    dummies = pd.get_dummies(values, prefix=col, drop_first=False, dtype=int)
    new_cols = [c for c in dummies.columns if c not in df.columns]
    if new_cols:
        df = pd.concat([df, dummies[new_cols]], axis=1)
    return df


def engineer_features_full(df_input: pd.DataFrame, name: str) -> pd.DataFrame:
    df = df_input.copy()

    for col in [
        "precio", "superficie_construida_m2", "numero_dormitorios", "numero_banos",
        "latitud", "longitud", "planta", "precio_m2_raw", *DISTANCE_COLS,
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in ["numero_dormitorios", "numero_banos", "latitud", "longitud", *DISTANCE_COLS]:
        if col in df.columns:
            median = df[col].median()
            df[col] = df[col].fillna(median if not pd.isna(median) else 0.0)

    for col in ["es_exterior", "tiene_ascensor", "tiene_garaje", "obra_nueva"]:
        if col in df.columns:
            df[col] = normalize_boolean(df[col])

    tip = df.get("tipologia", pd.Series(index=df.index, dtype="string"))
    tip = tip.astype("string").str.lower().str.strip().fillna("")
    df["tipologia_unificada"] = np.where(tip.isin(UNIFAMILIAR_VALUES), "unifamiliar", "piso")
    es_piso = (df["tipologia_unificada"] == "piso").astype(int)
    df["es_exterior_piso"] = df.get("es_exterior", 0) * es_piso
    df["tiene_ascensor_piso"] = df.get("tiene_ascensor", 0) * es_piso

    df["planta_num"] = df["planta"].apply(parse_planta) if "planta" in df.columns else 0

    positive_price = df["precio"].where(df["precio"] > 0)
    positive_size = df["superficie_construida_m2"].where(df["superficie_construida_m2"] > 0)
    df["log_precio"] = np.log(positive_price)
    df["log_superficie_construida_m2"] = np.log(positive_size)

    df["precio_m2"] = df["precio"] / df["superficie_construida_m2"].replace(0, np.nan)
    df["precio_m2_municipio_media"] = df.groupby("municipio")["precio_m2"].transform("mean")

    df["ratio_dormitorios_superficie"] = df["numero_dormitorios"] / df["superficie_construida_m2"].replace(0, np.nan)
    df["ratio_banos_superficie"] = df["numero_banos"] / df["superficie_construida_m2"].replace(0, np.nan)
    df["interaccion_superficie_banos"] = df["superficie_construida_m2"] * df["numero_banos"]
    df["interaccion_planta_sin_ascensor_piso"] = df["planta_num"] * (1 - df.get("tiene_ascensor", 0)) * es_piso

    df["latitud_2"] = df["latitud"] ** 2
    df["longitud_2"] = df["longitud"] ** 2
    df["interaccion_latitud_longitud"] = df["latitud"] * df["longitud"]

    centro_lat = df.groupby("municipio")["latitud"].transform("mean")
    centro_lon = df.groupby("municipio")["longitud"].transform("mean")
    df["distancia_centro_municipio_km"] = haversine_km(df["latitud"], df["longitud"], centro_lat, centro_lon)

    inv_distances = pd.DataFrame(index=df.index)
    for col in DISTANCE_COLS:
        inv_distances[f"inv_{col}"] = 1 / (1 + df[col])
    df["score_cercania_servicios"] = inv_distances.mean(axis=1)

    df["superficie_construida_m2_2"] = df["superficie_construida_m2"] ** 2
    df["numero_banos_2"] = df["numero_banos"] ** 2
    df["numero_dormitorios_2"] = df["numero_dormitorios"] ** 2

    for col in ["tipologia_unificada", "municipio"]:
        df = add_dummies_preserving_source(df, col)

    mun_cols = [c for c in df.columns if c.startswith("municipio_")]
    rare_muns = []
    if mun_cols:
        mun_counts = df[mun_cols].sum(numeric_only=True)
        rare_muns = mun_counts[mun_counts < MIN_OBS_MUNICIPIO].index.tolist()
        df["municipio_otro"] = df[rare_muns].max(axis=1) if rare_muns else 0

    bool_cols = df.select_dtypes(include="bool").columns
    if len(bool_cols):
        df[bool_cols] = df[bool_cols].astype(int)

    print(f"{name}: shape final {df.shape}; filas conservadas {len(df):,}; municipios raros marcados {len(rare_muns)}")
    return df

---
## 4. Aplicar y exportar

In [5]:
rent_final = engineer_features_full(rent_df, "RENT")
sale_final = engineer_features_full(sale_df, "SALE")

rent_final.to_csv(RENT_OUT, index=False)
sale_final.to_csv(SALE_OUT, index=False)

print(f"Exportado: {RENT_OUT} ({len(rent_final):,} filas, {rent_final.shape[1]} columnas)")
print(f"Exportado: {SALE_OUT} ({len(sale_final):,} filas, {sale_final.shape[1]} columnas)")

RENT: shape final (674, 147); filas conservadas 674; municipios raros marcados 49
SALE: shape final (2532, 154); filas conservadas 2,532; municipios raros marcados 30
Exportado: /Users/sitomachucas/Documents/BezanillaSL/data/gold/streamlit_rent.csv (674 filas, 147 columnas)
Exportado: /Users/sitomachucas/Documents/BezanillaSL/data/gold/streamlit_sale.csv (2,532 filas, 154 columnas)


---
## 5. Comprobaciones rapidas

In [6]:
required_streamlit_cols = [
    "propertyCode", "thumbnail", "url", "price", "priceByArea", "propertyType", "operation",
    "size", "rooms", "bathrooms", "address", "municipality", "district",
    "status", "newDevelopment", "parkingSpace.hasParkingSpace",
    "suggestedTexts.title", "suggestedTexts.subtitle",
]
required_ml_cols = [
    "log_precio", "log_superficie_construida_m2", "precio_m2", "precio_m2_municipio_media",
    "distancia_min_playa_km", "distancia_min_supermercado_km", "distancia_min_colegio_km",
    "distancia_centro_municipio_km", "score_cercania_servicios",
    "tipologia_unificada_piso", "tipologia_unificada_unifamiliar",
]

for name, df in [("RENT", rent_final), ("SALE", sale_final)]:
    missing_streamlit = [c for c in required_streamlit_cols if c not in df.columns]
    missing_ml = [c for c in required_ml_cols if c not in df.columns]
    print(f"{name}: missing Streamlit cols = {missing_streamlit}")
    print(f"{name}: missing ML cols = {missing_ml}")
    print(f"{name}: filas = {len(df):,}; columnas = {df.shape[1]:,}\n")

RENT: missing Streamlit cols = []
RENT: missing ML cols = []
RENT: filas = 674; columnas = 147

SALE: missing Streamlit cols = []
SALE: missing ML cols = []
SALE: filas = 2,532; columnas = 154

